In [1]:
# Run from anywhere: every path below is relative to the project root.
import os
if os.path.basename(os.getcwd()) == "scripts":
    os.chdir("..")

import pandas as pd

# Nargund 2015 Tables S1-S2 is the gene-level ATFS-1-bound list cited in the paper's
# prose. We need it to reconcile against Soo's binding column.
nargund_df = pd.read_excel("data/raw/nargund2015_TableS1-S2.xlsx")

print(f"Nargund 2015 bound-gene table: {nargund_df.shape[0]} rows, {nargund_df.shape[1]} columns")

# Each gene is searched under its public name, sequence name, AND any historical
# alias, since WormBase nomenclature has changed since 2015 and this table predates
# the renaming. ymel-1 was originally deposited as "yme-1" (after yeast YME1) - a
# search on "ymel-1"/"M03C11.5" alone silently misses it (see gate_decisions.md,
# "Binding Reconciliation Decision, corrected"). Confirmed independently against the
# raw GSE63803 ChIP peaks: dnj-10 has no called peak within 5kb of its TSS anywhere,
# while ymel-1 has one directly at its TSS (fold-enrichment 13.08, FDR 4%), also
# originally annotated "yme-1" by the depositors.
target_terms = {
    "hsp-6": ["hsp-6", "mthsp-70", "C37H5.8"],
    "hsp-60": ["hsp-60", "Y22D7AL.5"],
    "dnj-10": ["dnj-10", "F22B7.5"],
    "ymel-1": ["ymel-1", "yme-1", "M03C11.5"]
}

print("\n--- Nargund 2015 matches ---")
for gene_label, terms in target_terms.items():
    found = False
    for term in terms:
        matches = nargund_df.apply(lambda col: col.astype(str).str.contains(f"^{term}$|\\b{term}\\b", case=False, regex=True, na=False)).any(axis=1).sum()
        if matches > 0:
            print(f"{gene_label:8s} (found as '{term}'): {matches} match(es)")
            found = True
            break
    if not found:
        print(f"{gene_label:8s}: 0 match(es)")


Nargund 2015 bound-gene table: 511 rows, 7 columns

--- Nargund 2015 matches ---
hsp-6    (found as 'hsp-6'): 1 match(es)
hsp-60   (found as 'hsp-60'): 1 match(es)
dnj-10  : 0 match(es)
ymel-1   (found as 'yme-1'): 1 match(es)


/opt/homebrew/Caskroom/miniforge/base/envs/atfs1/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


In [2]:
# Soo's target table. Gene rows occupy rows 2-65 of the source sheet; everything
# past that is the legend, so it is trimmed off here.
soo_df = pd.read_excel("data/raw/ATFS1_targets_Soo.xlsx", sheet_name="Sheet1")
soo_df = soo_df.iloc[0:64].dropna(subset=["Gene name"])

soo_df = soo_df.rename(columns={
    "ATFS-1 bound \nin ChIP-seq": "soo_bound",
    "Gene sequence \nname": "seqname",
})[["Gene name", "seqname", "Score", "Score/variability", "soo_bound"]]

# Soo & Van Raamsdonk 2021 (microPublication Biology, 10.17912/micropub.biology.000484)
# state explicitly: "neither hsp-6 nor hsp-60 were among the 61 genes" in the
# high-confidence list. The sheet still carries them as the first two rows, scored
# with the same formula, because they are the field-standard reporter genes and the
# paper uses them as a comparison point (see roadmap: "hsp-6 inserted: rank 43 of 62").
# They are reference rows, not census members - split them out explicitly so nothing
# downstream silently treats this as a 63-gene set.
REFERENCE_GENES = ["hsp-6", "hsp-60"]
census_df = soo_df[~soo_df["Gene name"].isin(REFERENCE_GENES)].reset_index(drop=True)
reference_df = soo_df[soo_df["Gene name"].isin(REFERENCE_GENES)]

print(f"Soo table: {len(soo_df)} rows total")
print(f"  -> {len(census_df)} high-confidence census genes")
print(f"  -> {len(reference_df)} reference genes (hsp-6, hsp-60), excluded from the census")
if len(census_df) != 61:
    raise RuntimeError(f"Expected 61 census genes per Soo & Van Raamsdonk 2021, got {len(census_df)}.")

# Raw GSE63803 peaks. The first 23 lines are MACS run notes, not data.
peaks_df = pd.read_csv("data/raw/GSE63803_peaks.txt.gz", sep="\t", skiprows=23)

print("\nPeak columns:", peaks_df.columns.tolist())
print(f"Raw peak count in GSE63803: {len(peaks_df)} peaks")

# Does Soo's binding column agree with Nargund 2015 for the four chaperone/QC genes?
# GENE_ALIASES covers cases where WormBase nomenclature changed since 2015 - ymel-1
# was originally deposited as "yme-1" (after yeast YME1), and a plain-text search on
# the current name alone silently misses it (see gate_decisions.md, "Binding
# Reconciliation Decision, corrected"). Checked and confirmed against the raw
# GSE63803 peaks too: ymel-1 has a real called peak (fold-enrichment 13.08, FDR 4%)
# directly at its TSS, also originally annotated "yme-1"; dnj-10 has no peak within
# 5kb of its TSS at all, on either name.
nargund_text = nargund_df.astype(str).to_string()
GENE_ALIASES = {"ymel-1": ["ymel-1", "yme-1"]}
def is_nargund_bound(row):
    terms = GENE_ALIASES.get(str(row["Gene name"]), [str(row["Gene name"])])
    return any(t in nargund_text for t in terms) or str(row["seqname"]) in nargund_text
soo_df["nargund_bound"] = soo_df.apply(is_nargund_bound, axis=1)

target_genes = ["hsp-6", "hsp-60", "dnj-10", "ymel-1"]
comparison = soo_df[soo_df["Gene name"].isin(target_genes)][
    ["Gene name", "soo_bound", "nargund_bound"]
]

print("\n--- Binding: Soo column L vs. Nargund 2015 ---")
print(comparison.to_string(index=False))


Soo table: 63 rows total
  -> 61 high-confidence census genes
  -> 2 reference genes (hsp-6, hsp-60), excluded from the census

Peak columns: ['chr', 'start', 'end', 'length', 'summit', 'tags', '#NAME?', 'fold_enrichment', 'FDR(%)', 'Gene Name', 'Count']
Raw peak count in GSE63803: 1006 peaks

--- Binding: Soo column L vs. Nargund 2015 ---
Gene name soo_bound  nargund_bound
    hsp-6       Yes           True
   hsp-60       Yes           True
   dnj-10        No          False
   ymel-1        No           True


In [3]:
# Why skiprows=23 above: the peak file opens with MACS run notes. This prints the
# boundary between those notes and the header so the number is not a magic constant.
import gzip

with gzip.open("data/raw/GSE63803_peaks.txt.gz", "rt") as f:
    for i in range(1, 31):
        line = f.readline()
        if i >= 15:
            print(f"Line {i:2d}: {repr(line)}")


Line 15: '# total tags in treatment: 18329145\n'
Line 16: '# tags after filtering in treatment: 10992796\n'
Line 17: '# maximum duplicate tags at the same position in treatment = 1\t\t\t\t\t\t\t\t\t\t\n'
Line 18: '# Redundant rate in treatment: 0.40\t\t\t\t\t\t\t\t\t\t\n'
Line 19: '# total tags in control: 13281029\t\t\t\t\t\t\t\t\t\t\n'
Line 20: '# tags after filtering in control: 10583049\t\t\t\t\t\t\t\t\t\t\n'
Line 21: '# maximum duplicate tags at the same position in control = 1\t\t\t\t\t\t\t\t\t\t\n'
Line 22: '# Redundant rate in control: 0.20\t\t\t\t\t\t\t\t\t\t\n'
Line 23: '# d = 200\t\t\t\t\t\t\t\t\t\t\n'
Line 24: 'chr\tstart\tend\tlength\tsummit\ttags\t#NAME?\tfold_enrichment\tFDR(%)\tGene Name\tCount\n'
Line 25: 'chrI\t3031\t4483\t1453\t861\t299\t83.87\t3.26\t7.76\tY74C9A.3\t1\n'
Line 26: 'chrI\t107334\t112110\t4777\t3351\t1105\t651.11\t3.33\t6.72\trpl-7/rab11.1\t1\n'
Line 27: 'chrI\t124360\t125828\t1469\t538\t251\t73.57\t3.02\t7.59\tmex-1\t1\n'
Line 28: 'chrI\t172513\t174566

In [4]:
# Convert the raw ce6 peaks into a BED file for liftOver.
peaks_df = pd.read_csv("data/raw/GSE63803_peaks.txt.gz", sep="\t", skiprows=23)

# Trailing blank rows in the source would become NaN coordinates.
peaks_df = peaks_df.dropna(subset=["start", "end"]).copy()

bed_df = pd.DataFrame({
    "chr": peaks_df["chr"],
    "start": peaks_df["start"].astype("int64"),
    "end": peaks_df["end"].astype("int64"),
    "name": [f"peak_{i}" for i in range(len(peaks_df))],
})

os.makedirs("data/liftover", exist_ok=True)
bed_df.to_csv("data/liftover/peaks_ce6.bed", sep="\t", header=False, index=False)

print(f"Wrote {len(bed_df)} peaks to data/liftover/peaks_ce6.bed. First 3 lines:")
with open("data/liftover/peaks_ce6.bed") as f:
    for _ in range(3):
        print(f.readline().strip())


Wrote 1005 peaks to data/liftover/peaks_ce6.bed. First 3 lines:
chrI	3031	4483	peak_0
chrI	107334	112110	peak_1
chrI	124360	125828	peak_2


In [5]:
# Operon status for the two anchor genes (Gate 0, decision 2).
#
# The WormBase REST API returns 403 to scripted requests, so this is resolved against
# the local WS285 GFF3, where operons are annotated features carrying an explicit
# `genes=` membership list. Offline, versioned, reproducible. Any failure raises
# rather than falling through to a default answer.
#
# Position matters, not just membership: downstream operon members are trans-spliced
# off the operon head and have no promoter of their own, so a "no peak" call for them
# is an artefact of nearest-TSS assignment. An operon head keeps its own promoter.
import gzip

anchors = {"C07G1.7": "WBGene00015573", "F22B3.7": "WBGene00009038"}

operons = {}  # name -> ordered list of member WBGene IDs
with gzip.open("data/raw/c_elegans.PRJNA13758.WS285.annotations.gff3.gz", "rt") as fh:
    for line in fh:
        if line.startswith("#"):
            continue
        f = line.rstrip("\n").split("\t")
        # Column 2 separates live operons from deprecated_operon entries.
        if len(f) < 9 or f[1] != "operon" or f[2] != "operon":
            continue
        attrs = dict(
            kv.split("=", 1) for kv in f[8].split(";") if "=" in kv
        )
        name = attrs.get("Name")
        members = attrs.get("genes", "").split(",") if attrs.get("genes") else []
        if name and members:
            operons[name] = members

if not operons:
    raise RuntimeError("No live operon features parsed from the GFF3 - check the annotation file.")

print(f"WS285 GFF3: {len(operons)} live operons parsed")
print("\n--- Operon status (WS285 GFF3) ---")
for seqname, wbgene in anchors.items():
    hit = next(((n, m) for n, m in operons.items() if wbgene in m), None)
    if hit is None:
        print(f"{seqname} ({wbgene}): not in any operon - independently promoted")
    else:
        name, members = hit
        pos = members.index(wbgene) + 1
        role = "operon HEAD (keeps its own promoter)" if pos == 1 else "DOWNSTREAM member (no promoter of its own)"
        print(f"{seqname} ({wbgene}): in {name}, position {pos} of {len(members)} - {role}")


WS285 GFF3: 1385 live operons parsed

--- Operon status (WS285 GFF3) ---
C07G1.7 (WBGene00015573): not in any operon - independently promoted
F22B3.7 (WBGene00009038): not in any operon - independently promoted


In [6]:
# Gate 1's named-consequence question: is hsp-6 in Nargund 2012's ATFS-1-dependent
# spg-7 set? Table S3 is that set. Per the paper's Materials and Methods, a gene
# counts as ATFS-1-dependent if its up-regulation in atfs-1(tm4525) was <=25% of the
# up-regulation in wild-type, both raised on spg-7(RNAi) vs control(RNAi).
table_s3 = pd.read_excel(
    "data/raw/nargund2012_TableS3_spg7_ATFS1dependent.xlsx", sheet_name="Sheet1", header=None
)
table_s3.columns = [
    "seqname", "symbol", "function", "wt_fold", "atfs1_fold",
    "fold_diff", "blank", "pct_less", "na"
]

# A real gene row has a numeric wt_fold value; title, blank, subheader, the literal
# column-header row, and category-divider rows do not. Counting by symbol.notna()
# instead silently drops every gene that has no assigned public symbol (very common
# in C. elegans annotation - 229 of the 391 real rows here, including the paper's
# own anchor genes C07G1.7 and F22B3.7) and wrongly counts the header row itself as
# a gene. See gate_decisions.md, "Table S2/S3 row count correction."
table_s3["wt_fold_numeric"] = pd.to_numeric(table_s3["wt_fold"], errors="coerce")
table_s3_real = table_s3[table_s3["wt_fold_numeric"].notna()].copy()

hsp6_terms = ["hsp-6", "mthsp-70", "C37H5.8"]
hsp6_in_s3 = table_s3_real.apply(
    lambda row: any(str(t).lower() in str(row.values).lower() for t in hsp6_terms), axis=1
).any()

print(f"hsp-6 in Nargund 2012 Table S3 (spg-7 RNAi, ATFS-1-dependent set): {hsp6_in_s3}")
if not hsp6_in_s3:
    print("Checked by sequence name and gene symbol across all",
          f"{len(table_s3_real)} genes in Table S3; not found.")
    print("Source: Nargund et al. 2012, Science 337(6094):587-90, Supplementary Table S3.")


hsp-6 in Nargund 2012 Table S3 (spg-7 RNAi, ATFS-1-dependent set): False
Checked by sequence name and gene symbol across all 391 genes in Table S3; not found.
Source: Nargund et al. 2012, Science 337(6094):587-90, Supplementary Table S3.


/opt/homebrew/Caskroom/miniforge/base/envs/atfs1/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


In [7]:
# GSE110984 / GSE93724 overlap.
#
# This is a citation, not a computation: NCBI blocks scripted access to GEO, so the
# summary text below was read off the record by hand and is recorded here verbatim
# with its retrieval date. It is not re-derived at runtime.
GEO_ACCESSION = "GSE110984"
GEO_URL = "https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE110984"
GEO_SUMMARY_QUOTE = "Note that sequencing batch 2 was previously uploaded as part of GSE93724."
RETRIEVED = "2026-08-11 (manual retrieval; independently re-checked against the live record)"

print("--- GSE110984 / GSE93724 overlap ---")
print(f"Source     : {GEO_ACCESSION} summary, retrieved {RETRIEVED}")
print(f"URL        : {GEO_URL}")
print(f"Quote      : \"{GEO_SUMMARY_QUOTE}\"")
print("Consequence: batches overlap. Use GSE110984 as the primary series so the")
print("             shared batch-2 samples are not counted twice.")


--- GSE110984 / GSE93724 overlap ---
Source     : GSE110984 summary, retrieved 2026-08-11 (manual retrieval; independently re-checked against the live record)
URL        : https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE110984
Quote      : "Note that sequencing batch 2 was previously uploaded as part of GSE93724."
Consequence: batches overlap. Use GSE110984 as the primary series so the
             shared batch-2 samples are not counted twice.
